In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential, layers, optimizers
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img

In [ ]:
folder = Path("caltech101")
classes = sorted(d.name for d in folder.iterdir() if d.is_dir() and d.name not in ["BACKGROUND_Google", "train"])
data = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)
train = data.flow_from_directory(folder, classes=classes, target_size=(224, 224),
    class_mode="sparse", batch_size=16, subset="training")
test = data.flow_from_directory(folder, classes=classes, target_size=(224, 224),
    class_mode="sparse", batch_size=16, subset="validation", shuffle=False)

In [ ]:
base = VGG16(weights="vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5", include_top=False, pooling="avg")
base.trainable = False
model = Sequential([base, layers.Dense(len(classes), activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, epochs=3)

In [ ]:
base.trainable = True
for layer in base.layers[:-4]:
    layer.trainable = False
model.compile(optimizer=optimizers.Adam(0.00001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, epochs=1)

In [ ]:
print("Loss, accuracy:", model.evaluate(test))
prediction = model.predict(test[0][0][:1], verbose=0).argmax()
plt.imshow(load_img(test.filepaths[0]))
plt.title("Actual: " + classes[test.classes[0]] + " | Predicted: " + classes[prediction])
plt.show()